# Объединение данных и когортный анализ

В этом блокноте:
1. Добавляем столбец **Cohort** (месяц создания контакта) к Contacts, Deals и Spend
2. Объединяем все датасеты в единую аналитическую витрину
3. Готовим базу для расчёта юнит-экономики по когортам

**Логика когорт:**
- `Cohort` = месяц, в котором контакт был создан (`contacts.Created Time`)
- Сделки получают когорту через связь с контактом (`deals.Contact Name → contacts.Id`)
- Расходы получают когорту из даты транзакции (`spend.Date`), и привязываются по `Source`

In [12]:
import pandas as pd
import numpy as np
import os
import help_130625_dam as h

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

CLEANED_DIR = os.path.join('..', 'data', 'cleaned')

## Загрузка очищенных данных

In [13]:
# Загружаем все .pkl файлы
contacts = pd.read_pickle(os.path.join(CLEANED_DIR, 'contacts_clean.pkl'))
deals = pd.read_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
calls = pd.read_pickle(os.path.join(CLEANED_DIR, 'calls_clean.pkl'))
spend = pd.read_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))

print(f"Contacts: {contacts.shape}")
print(f"Deals: {deals.shape}")
print(f"Calls: {calls.shape}")
print(f"Spend: {spend.shape}")

Contacts: (18548, 5)
Deals: (21591, 26)
Calls: (95874, 10)
Spend: (19862, 9)


## Когорты

**Когорта** = год-месяц, когда лид впервые был зарегистрирован.  

In [14]:
contacts['cohort'] = contacts['created_time'].dt.to_period('M')

print('Contacts: добавлен столбец Cohort')
print(f'Диапазон когорт: {contacts["cohort"].min()} → {contacts["cohort"].max()}')
print(f'\nКонтактов по когортам:')
display(contacts['cohort'].value_counts().sort_index().to_frame('Contacts'))


Contacts: добавлен столбец Cohort
Диапазон когорт: 2023-06 → 2024-06

Контактов по когортам:


,Contacts
cohort,
2023-06,1
2023-07,630
2023-08,899
2023-09,992
2023-10,1458
2023-11,1576
2023-12,1668
2024-01,1992
2024-02,1730


In [15]:
cohort_map = contacts.set_index('id')['cohort']
deals['cohort'] = deals['contact_id'].map(cohort_map)

matched = deals['cohort'].notna().sum()
no_cohort = deals['cohort'].isna().sum()
print(f'Deals: добавлен столбец cohort')
print(f'Сделок с когортой:      {matched:,} ({matched/len(deals)*100:.1f}%)')
print(f'Сделок без когорты:     {no_cohort:,}  (нет в contacts — оставляем)')
print(f'\nДиапазон когорт: {deals["cohort"].dropna().min()} → {deals["cohort"].dropna().max()}')
print(f'\nСделок по когортам:')
display(deals['cohort'].value_counts().sort_index().to_frame('Deals'))


Deals: добавлен столбец cohort
Сделок с когортой:      21,529 (99.7%)
Сделок без когорты:     62  (нет в contacts — оставляем)

Диапазон когорт: 2023-07 → 2024-06

Сделок по когортам:


,Deals
cohort,
2023-07,974
2023-08,1282
2023-09,1254
2023-10,1865
2023-11,1932
2023-12,1927
2024-01,2254
2024-02,1985
2024-03,2227


In [16]:
# Расходы привязываем к месяцу, в котором они были понесены
spend['cohort'] = spend['date'].dt.to_period('M')

print('Spend: добавлен столбец cohort')
print(f'Диапазон когорт: {spend["cohort"].min()} → {spend["cohort"].max()}')

# Агрегация расходов: source + cohort
# Это основная таблица затрат для юнит-экономики
# Теперь включаем показы, клики и расходы для воронки "Показ -> Клик -> Лид"
spend_by_source_cohort = (
    spend.groupby(['source', 'cohort'], observed=True)
    .agg(
        spend_total  = ('spend', 'sum'),
        clicks       = ('clicks', 'sum'),
        impressions  = ('impressions', 'sum')
    )
    .reset_index()
)

print(f'\nРасходы по source + cohort: {spend_by_source_cohort.shape[0]} строк')
display(spend_by_source_cohort.sort_values(['cohort', 'spend_total'], ascending=[True, False]).head(15))


Spend: добавлен столбец cohort
Диапазон когорт: 2023-07 → 2024-06

Расходы по source + cohort: 133 строк


,source,cohort,spend_total,clicks,impressions
36,Google Ads,2023-07,2630.93,16285,1331036
24,Facebook Ads,2023-07,1588.27,2185,117335
98,Tiktok Ads,2023-07,651.54,975,203083
121,Youtube Ads,2023-07,587.58,769,134682
83,Telegram posts,2023-07,399.00,1093,57757
0,Bloggers,2023-07,205.00,268,24919
12,CRM,2023-07,0.00,0,0
50,Organic,2023-07,0.00,1832,0
71,SMM,2023-07,0.00,146,0
37,Google Ads,2023-08,3669.30,23691,2666971


## 3. Объединение Contacts + Deals

Один контакт может иметь несколько сделок. Мы делаем `left join`:  
- все контакты попадают в витрину (даже без сделок — это лиды)
- для каждого контакта агрегируем ключевые показатели по сделкам

In [25]:

# Агрегация сделок по контакту для join
deals_agg = (
    deals.groupby('contact_id', dropna=True)
    .agg(
        deals_count           = ('id', 'count'),
        deals_won             = ('stage_group', lambda x: (x == 'Won/Paid').sum()),
        source                = ('source', 'first'),
        campaign              = ('campaign', 'first'),
        offer_total           = ('offer_total_amount', 'max'),
        deal_created_first    = ('created_time', 'min'),
        deal_closed_last      = ('closing_date', 'max'),
        sla_min               = ('sla', 'min'),
        product               = ('product', 'first'),
        education_type        = ('education_type', 'first'),
        city                  = ('city', 'first'),
        level_of_deutsch      = ('level_of_deutsch', 'last'),
        course_duration       = ('course_duration', 'last'),
        stage_group           = ('stage_group', 'last'),
        quality               = ('quality', 'last'),
    )
    .reset_index()
    .rename(columns={'contact_id': 'id'})
)

print(f'Сгруппировано → {len(deals_agg):,} уникальных контактов со сделками')
display(deals_agg.head())


Сгруппировано → 18,090 уникальных контактов со сделками


,id,deals_count,deals_won,source,campaign,offer_total,deal_created_first,deal_closed_last,sla_min,product,education_type,city,level_of_deutsch,course_duration,stage_group,quality
0,-1,61,10,Organic,nina,11500.00,2023-07-13 10:01:00,2024-05-31,4,Unknown,Unknown,-,B1,6.00,Lost,E - Non Qualified
1,5805028000000872003,11,0,Facebook Ads,04.07.23recentlymoved_DE,4000.00,2023-07-25 10:41:00,2023-08-07,2546,Unknown,Unknown,Unknown,A2,11.00,Marketing/Lead,Unknown
2,5805028000000889001,2,0,Organic,Unknown,0.00,2023-08-18 21:29:00,2023-10-26,2546,Unknown,Unknown,Unknown,Unknown,NaN,Lost,E - Non Qualified
3,5805028000000907006,2,0,Organic,Unknown,0.00,2023-11-23 10:48:00,2023-12-29,14344,Unknown,Unknown,Unknown,Unknown,NaN,Lost,E - Non Qualified
4,5805028000000939010,2,1,Facebook Ads,02.07.23wide_DE,11000.00,2023-07-04 10:11:00,2024-03-01,4895,Digital Marketing,Morning,Hamburg,Unknown,11.00,Lost,D - Non Target


In [26]:

# LEFT JOIN: Contacts + агрегированные Deals
master = contacts.merge(deals_agg, on='id', how='left')

# Флаг: есть ли у контакта хотя бы одна сделка
master['has_deal'] = master['deals_count'].notna()

# Contacts без сделок — лиды (0 сделок)
master['deals_count'] = master['deals_count'].fillna(0).astype('int32')
master['deals_won']   = master['deals_won'].fillna(0).astype('int32')

display(master.head())

print(f'master: {master.shape}')
print(f'Полная оплата (deals_won>0): {(master["deals_won"]>0).sum():,} — только Payment Done')
print(f'Контактов со сделками:  {master["has_deal"].sum():,} ({master["has_deal"].mean()*100:.1f}%)')
print(f'Контактов без сделок:   {(~master["has_deal"]).sum():,}')


,id,contact_owner_name,created_time,modified_time,first_payment_date,cohort,deals_count,deals_won,source,campaign,offer_total,deal_created_first,deal_closed_last,sla_min,product,education_type,city,level_of_deutsch,course_duration,stage_group,quality,has_deal
0,5805028000000645014,Rachel White,2023-06-27 11:28:00,2023-12-22 13:34:00,NaT,2023-06,0,0,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,5805028000000872003,Charlie Davis,2023-07-03 11:31:00,2024-05-21 10:23:00,2023-07-25 10:41:00,2023-07,11,0,Facebook Ads,04.07.23recentlymoved_DE,4000.00,2023-07-25 10:41:00,2023-08-07,2546,Unknown,Unknown,Unknown,A2,11.00,Marketing/Lead,Unknown,True
2,5805028000000889001,Bob Brown,2023-07-02 22:37:00,2023-12-21 13:17:00,NaT,2023-07,2,0,Organic,Unknown,0.00,2023-08-18 21:29:00,2023-10-26,2546,Unknown,Unknown,Unknown,Unknown,NaN,Lost,E - Non Qualified,True
3,5805028000000907006,Bob Brown,2023-07-03 05:44:00,2023-12-29 15:20:00,NaT,2023-07,2,0,Organic,Unknown,0.00,2023-11-23 10:48:00,2023-12-29,14344,Unknown,Unknown,Unknown,Unknown,NaN,Lost,E - Non Qualified,True
4,5805028000000939010,Nina Scott,2023-07-04 10:11:00,2024-04-16 16:14:00,2023-07-04 10:11:00,2023-07,2,1,Facebook Ads,02.07.23wide_DE,11000.00,2023-07-04 10:11:00,2024-03-01,4895,Digital Marketing,Morning,Hamburg,Unknown,11.00,Lost,D - Non Target,True


master: (18548, 22)
Полная оплата (deals_won>0): 845 — только Payment Done
Контактов со сделками:  18,088 (97.5%)
Контактов без сделок:   460


## 4. Добавление звонков (Calls)

Агрегируем по `CONTACTID`: количество звонков, успешных, суммарная длительность.

In [27]:
# Агрегация звонков по contactid
calls_agg = (
    calls.groupby('contactid', dropna=True)
    .agg(
        calls_total         = ('id', 'count'),
        calls_successful    = ('is_successful', 'sum'),
        call_duration_total = ('call_duration_in_seconds', 'sum'),
        call_duration_avg   = ('call_duration_in_seconds', 'mean'),
        first_call_date     = ('call_start_time', 'min'),
        last_call_date      = ('call_start_time', 'max'),
    )
    .reset_index()
    .rename(columns={'contactid': 'id'})
)

calls_agg['calls_success_rate'] = (calls_agg['calls_successful'] / calls_agg['calls_total']).round(4)

# JOIN с master
master = master.merge(calls_agg, on='id', how='left')

# Заполняем 0 для контактов без звонков
for col in ['calls_total', 'calls_successful', 'call_duration_total']:
    master[col] = master[col].fillna(0).astype('int32')
master['call_duration_avg'] = master['call_duration_avg'].fillna(0).round(1)

print(f'master после добавления звонков: {master.shape}')
print(f'Контактов с хотя бы 1 звонком: {(master["calls_total"] > 0).sum():,}')
display(master[['id', 'cohort', 'has_deal', 'calls_total', 'calls_successful', 'calls_success_rate']].head(10))

master после добавления звонков: (18548, 29)
Контактов с хотя бы 1 звонком: 15,214


,id,cohort,has_deal,calls_total,calls_successful,calls_success_rate
0,5805028000000645014,2023-06,False,8,8,1.00
1,5805028000000872003,2023-07,True,12,1,0.08
2,5805028000000889001,2023-07,True,0,0,<NA>
3,5805028000000907006,2023-07,True,0,0,<NA>
4,5805028000000939010,2023-07,True,18,18,1.00
5,5805028000000942003,2023-07,True,14,12,0.86
6,5805028000000961001,2023-07,True,6,1,0.17
7,5805028000000964025,2023-07,True,28,24,0.86
8,5805028000000964068,2023-07,True,6,6,1.00
9,5805028000000968001,2023-07,True,3,2,0.67


## 6. Сохранение

In [28]:
os.makedirs(os.path.join('..', 'data', 'cleaned'), exist_ok=True)

# Основная витрина: 1 строка = 1 контакт (лид) со всеми данными сделок и звонков
master.to_pickle(os.path.join(CLEANED_DIR, 'master_clean.pkl'))

# Сохраняем также spend и deals с добавленными когортами для аналитики
deals.to_pickle(os.path.join(CLEANED_DIR, 'deals_clean.pkl'))
spend.to_pickle(os.path.join(CLEANED_DIR, 'spend_clean.pkl'))

print(f'master_clean.pkl: {master.shape} (витрина готова)')
print(f'deals_clean.pkl:  {deals.shape}  (когорты добавлены)')
print(f'spend_clean.pkl:  {spend.shape}  (когорты добавлены)')

print(f'\nmaster — столбцы:')
print(list(master.columns))

master_clean.pkl: (18548, 29) (витрина готова)
deals_clean.pkl:  (21591, 26)  (когорты добавлены)
spend_clean.pkl:  (19862, 9)  (когорты добавлены)

master — столбцы:
['id', 'contact_owner_name', 'created_time', 'modified_time', 'first_payment_date', 'cohort', 'deals_count', 'deals_won', 'source', 'campaign', 'offer_total', 'deal_created_first', 'deal_closed_last', 'sla_min', 'product', 'education_type', 'city', 'level_of_deutsch', 'course_duration', 'stage_group', 'quality', 'has_deal', 'calls_total', 'calls_successful', 'call_duration_total', 'call_duration_avg', 'first_call_date', 'last_call_date', 'calls_success_rate']


## Описание датасета Master (Витрина данных)

**Источник:** Сводная таблица, объединяющая данные из `contacts`, `deals`, `calls` и `spend`.  
**Назначение:** Единый источник истины для аналитики. Позволяет рассчитывать ROMI, конверсии в продажи, эффективность звонков и стоимость привлечения в разрезе когорт и источников.

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID контакта (основной ключ витрины) |
| `cohort` | `period` | **Когорта:** месяц регистрации контакта (напр. 2024-06) |
| `source` | `category` | Маркетинговый источник контакта |
| `deal_count` | `int64` | Общее количество сделок, связанных с этим контактом |
| `total_paid` | `float64` | **LTV:** Суммарная выручка (оплаты) от этого контакта |
| `call_count` | `int64` | Количество звонков контакту |
| `is_buyer` | `bool` | Флаг: совершил ли контакт хотя бы одну оплату |
| `avg_sla_seconds` | `float64` | Среднее время ответа менеджера (в секундах) |

### Агрегированные данные (справочно)
*Используются для расчета ROI на уровне сегментов:*
- `spend_cohort_source` — затраты на привлечение когорты через конкретный источник.
- `conversion_rate` — расчетный показатель эффективности воронки.

**Ключевая логика сборки:**
1. За основу берутся все **контакты** (`01_cleaning_contacts`).
2. Добавляется **когорта** на основе даты регистрации.
3. Джойнятся **агрегаты по сделкам** (`03_cleaning_deals`): количество и сумма оплат.
4. Джойнятся **агрегаты по звонкам** (`04_cleaning_calls`): количество и результативность.
5. Присоединяются **затраты** (`02_cleaning_spend`), распределенные по источникам и датам.
